# Raw data

Sections 1–4 of the forecasting pipeline. Run this notebook from `notebooks/`.
It writes `output/Tetouan_clean_10min.csv` for the processed-data notebook.


## 1. Libraries and Configurations

Notebooks use popular libraries and do not require installation of a machine learning model at this stage.

In [9]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid", context="notebook")
VC_RED = "#D71920"
VC_DARK = "#8B1014"
ZONE_COLORS = ["#D71920", "#4C78A8", "#59A14F"]

ModuleNotFoundError: No module named 'matplotlib'

## 2. Read Data

In [ ]:
DATA_PATH = "Tetuan City power consumption.csv"

df_raw = pd.read_csv(DATA_PATH)

print(f"Data size {df_raw.shape[0]:,} lines {df_raw.shape[1]} post")
display(df_raw.head())

Original data size: 52,416 lines × 9 columns


,DateTime,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
0,1/1/2017 0:00,6.56,73.80,0.08,0.05,0.12,"34,055.70","16,128.88","20,240.96"
1,1/1/2017 0:10,6.41,74.50,0.08,0.07,0.09,"29,814.68","19,375.08","20,131.08"
2,1/1/2017 0:20,6.31,74.50,0.08,0.06,0.10,"29,128.10","19,006.69","19,668.43"
3,1/1/2017 0:30,6.12,75.00,0.08,0.09,0.10,"28,228.86","18,361.09","18,899.28"
4,1/1/2017 0:40,5.92,75.70,0.08,0.05,0.09,"27,335.70","17,872.34","18,442.41"


## 3. Data Understanding and Normalization of Column Names

In [ ]:
df = df_raw.copy()

df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

df = df.rename(columns={
    "zone_1_power_consumption": "zone_1_power",
    "zone_2_power_consumption": "zone_2_power",
    "zone_3_power_consumption": "zone_3_power",
})

required_columns = {
    "datetime", 
    "temperature", 
    "humidity", 
    "wind_speed",
    "general_diffuse_flows", 
    "diffuse_flows",
    "zone_1_power", 
    "zone_2_power", 
    "zone_3_power"
}

missing_columns = sorted(required_columns - set(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["datetime"] = pd.to_datetime(
    df["datetime"], format="%m/%d/%Y %H:%M", errors="raise"
)

df = df.sort_values("datetime").reset_index(drop=True)

zone_cols = ["zone_1_power", "zone_2_power", "zone_3_power"]
weather_cols = [
    "temperature", "humidity", "wind_speed",
    "general_diffuse_flows", "diffuse_flows"
]
numeric_cols = weather_cols + zone_cols

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="raise")

print("Post-standardized columns:")
print(df.columns.tolist())
print(f"Interview time duration: {df['datetime'].min()} to {df['datetime'].max()}")
display(df.dtypes.to_frame("dtype"))

Post-standardized columns:
['datetime', 'temperature', 'humidity', 'wind_speed', 'general_diffuse_flows', 'diffuse_flows', 'zone_1_power', 'zone_2_power', 'zone_3_power']
Period: 2017-01-01 00:00:00 to 2017-12-30 23:50:00


,dtype
datetime,datetime64[us]
temperature,float64
humidity,float64
wind_speed,float64
general_diffuse_flows,float64
diffuse_flows,float64
zone_1_power,float64
zone_2_power,float64
zone_3_power,float64


## 4. Checking and cleaning data

Mandatory tests:

- No error or duplicate timestamps.
- There are no missing values in the required fields.
- The distance between two consecutive records is 10 minutes.
- Non-negative power consumption.
- Humidity is in the range of 0–100.

Peak demand is not automatically deleted as this is an important signal for capacity planning.

In [ ]:
duplicate_rows = int(df.duplicated().sum())
duplicate_timestamps = int(df["datetime"].duplicated().sum())
missing_by_column = df[list(required_columns)].isna().sum()
time_diff = df["datetime"].diff().dropna()
irregular_intervals = int((time_diff != pd.Timedelta(minutes=10)).sum())
negative_power = int((df[zone_cols] < 0).sum().sum())
invalid_humidity = int((~df["humidity"].between(0, 100)).sum())

quality_report = pd.Series({
    "Line Id": len(df),
    "Full line overlap": duplicate_rows,
    "Duplicate timestamp": duplicate_timestamps,
    "Total missing values": int(missing_by_column.sum()),
    "Elapsed time interval of 10 minutes": irregular_intervals,
    "Negative power value": negative_power,
    "External Humidity [0, 100]": invalid_humidity,
}, name="results")

display(quality_report.to_frame())
display(missing_by_column.to_frame("missing_count"))

assert duplicate_timestamps == 0, "Duplicate timestamp detected"
assert missing_by_column.sum() == 0, "Missing values detected"
assert irregular_intervals == 0, "Intermittent time series according to step 10 min"
assert negative_power == 0, "Negative power consumption detected"
assert invalid_humidity == 0, "Humidity is outside the 0–100 range"

if duplicate_rows > 0:
    df = df.drop_duplicates().reset_index(drop=True)

print("Data pass required quality checks")

,results
Line Id,52416
Full line overlap,0
Duplicate timestamp,0
Total missing values,0
Elapsed time interval of 10 minutes,0
Negative power value,0
"External Humidity [0, 100]",0


,missing_count
wind_speed,0
diffuse_flows,0
datetime,0
temperature,0
zone_3_power,0
zone_1_power,0
humidity,0
general_diffuse_flows,0
zone_2_power,0


Data pass required quality checks


In [ ]:
descriptive_stats = df[numeric_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

display(descriptive_stats)

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
temperature,"52,416.00",18.81,5.82,3.25,6.76,9.80,14.41,18.78,22.89,28.39,33.16,40.01
humidity,"52,416.00",68.26,15.55,11.34,27.33,39.45,58.31,69.86,81.40,88.90,92.00,94.80
wind_speed,"52,416.00",1.96,2.35,0.05,0.07,0.07,0.08,0.09,4.92,4.92,4.93,6.48
general_diffuse_flows,"52,416.00",182.70,264.40,0.00,0.02,0.04,0.06,5.04,319.60,782.00,885.00,"1,163.00"
diffuse_flows,"52,416.00",75.03,124.21,0.01,0.06,0.09,0.12,4.46,101.00,331.85,596.80,936.00
zone_1_power,"52,416.00","32,344.97","7,130.56","13,895.70","20,274.81","21,867.34","26,310.67","32,265.92","37,309.02","44,712.06","48,099.48","52,204.40"
zone_2_power,"52,416.00","21,042.51","5,201.47","8,560.08","11,631.70","13,284.15","16,980.77","20,823.17","24,713.72","30,387.14","33,734.95","37,408.86"
zone_3_power,"52,416.00","17,835.41","6,622.17","5,935.17","7,726.27","9,519.33","13,129.33","16,415.12","21,624.10","29,905.71","38,722.44","47,598.33"


## Export the cleaned 10-minute series


In [ ]:
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
clean_path = output_dir / "Tetouan_clean_10min.csv"
df.to_csv(clean_path, index=False)
print(f"Saved {clean_path}")
